# UMUD Challenge - Submission 4: K-Fold Ensemble + Stronger Architecture

**Key upgrades over Submission 3:**
1. **EfficientNet-B3 encoder** (much stronger feature extraction than ResNet34)
2. **Dice + BCE combined loss** (the gold standard for medical image segmentation)
3. **3-Fold Cross-Validation Ensemble** (train 3 models per task, average their predictions)
4. **512x512 resolution** (captures the finest fascicle details)
5. **ReduceLROnPlateau scheduler** (adapts learning rate when validation plateaus)
6. **Heavier augmentation** (ElasticTransform, GridDistortion for ultrasound deformation)
7. **5x TTA** (original + hflip + vflip + bright + dark)
8. **All warnings suppressed**

**Instructions:** GPU ON, Internet ON, Run All.

In [ ]:
%%capture
# %%capture suppresses ALL output from pip install (no warnings!)
!pip install segmentation-models-pytorch albumentations


In [ ]:
import os, sys, warnings, logging
# Aggressively suppress ALL warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['PYTHONWARNINGS'] = 'ignore'
warnings.filterwarnings('ignore')
logging.disable(logging.WARNING)

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from pathlib import Path
import glob
from tqdm.auto import tqdm
import math

import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

# --- Path Discovery (same pattern as Submission 2/3 that worked for you) ---
matching_dirs = glob.glob('/kaggle/input/**/apo_imgs_v1', recursive=True)
if matching_dirs:
    APO_IMG_DIR = Path(matching_dirs[0])
    COMPETITION_DIR = APO_IMG_DIR.parent
else:
    COMPETITION_DIR = Path(glob.glob('/kaggle/input/*')[0]) if glob.glob('/kaggle/input/*') else Path('.')

APO_IMG_DIR   = COMPETITION_DIR / 'apo_imgs_v1' / 'apo_images_new_model_v1'
APO_MASK_DIR  = COMPETITION_DIR / 'apo_masks_v1' / 'apo_masks_new_model_v1'
FASC_IMG_DIR  = COMPETITION_DIR / 'fasc_imgs_v1' / 'fasc_images_new_model_v1'
FASC_MASK_DIR = COMPETITION_DIR / 'fasc_masks_v1' / 'fasc_masks_new_model_v1'
TEST_DIR      = COMPETITION_DIR / 'test_images_v2' / 'test_set_v2'
OUT_DIR       = Path('/kaggle/working')

print("--- Path Check ---")
for name, p in [("APO_IMG", APO_IMG_DIR), ("APO_MASK", APO_MASK_DIR),
                ("FASC_IMG", FASC_IMG_DIR), ("FASC_MASK", FASC_MASK_DIR),
                ("TEST", TEST_DIR)]:
    print(f"  {name}: {p} -> Exists: {p.exists()}")

IMG_SIZE = 512
BATCH_SIZE = 4        # Smaller batch for 512x512 on Kaggle GPU
EPOCHS_PER_FOLD = 10  # 10 epochs x 3 folds x 2 models = 60 total epochs
N_FOLDS = 3
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SCALE_MM_PER_PIXEL = 0.06979

print(f"\nDevice: {DEVICE}")
print(f"Config: {IMG_SIZE}px, BS={BATCH_SIZE}, {EPOCHS_PER_FOLD} epochs/fold, {N_FOLDS} folds")


In [ ]:
# --- Augmentation pipelines ---
train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.4),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, 
                       border_mode=cv2.BORDER_CONSTANT, p=0.5),
    A.ElasticTransform(alpha=30, sigma=5, p=0.2),
    A.GridDistortion(num_steps=5, distort_limit=0.1, p=0.2),
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
    A.Normalize(mean=[0.5], std=[0.5]),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.5], std=[0.5]),
    ToTensorV2()
])

class SingleMaskDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None):
        self.img_dir = Path(img_dir)
        self.mask_dir = Path(mask_dir)
        self.transform = transform
        self.images = sorted([p for p in self.img_dir.rglob('*') 
                              if p.is_file() and p.suffix.lower() in ['.tif', '.png', '.jpg']])
        # Pre-build a stem->mask_path lookup to avoid slow rglob per item
        self._mask_lookup = {}
        for p in self.mask_dir.rglob('*'):
            if p.is_file() and p.suffix.lower() in ['.tif', '.png', '.jpg']:
                self._mask_lookup[p.stem] = p

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        mask_path = self._mask_lookup.get(img_path.stem, None)

        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        if img is None:
            img = np.zeros((800, 1200), dtype=np.uint8)

        if mask_path:
            mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
            if mask is None:
                mask = np.zeros(img.shape[:2], dtype=np.uint8)
            elif mask.shape[:2] != img.shape[:2]:
                mask = cv2.resize(mask, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)
        else:
            mask = np.zeros(img.shape[:2], dtype=np.uint8)

        mask = (mask > 127).astype(np.float32)

        if self.transform:
            augmented = self.transform(image=img, mask=mask)
            img = augmented['image']       # (1, H, W)
            mask = augmented['mask'].unsqueeze(0)  # (1, H, W)
        else:
            img = torch.from_numpy(img).unsqueeze(0).float() / 255.0
            mask = torch.from_numpy(mask).unsqueeze(0).float()

        return img, mask

# Build full datasets (no transform yet - we assign per fold)
apo_full = SingleMaskDataset(APO_IMG_DIR, APO_MASK_DIR)
fasc_full = SingleMaskDataset(FASC_IMG_DIR, FASC_MASK_DIR)
print(f"Apo dataset: {len(apo_full)} images")
print(f"Fasc dataset: {len(fasc_full)} images")


In [ ]:
# --- Dice + BCE Combined Loss ---
class DiceBCELoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
    
    def forward(self, pred, target):
        bce_loss = self.bce(pred, target)
        pred_sig = torch.sigmoid(pred)
        smooth = 1.0
        intersection = (pred_sig * target).sum(dim=(2, 3))
        union = pred_sig.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
        dice_loss = 1.0 - (2.0 * intersection + smooth) / (union + smooth)
        return bce_loss + dice_loss.mean()

# --- K-Fold Training Function ---
def train_kfold(full_dataset, model_prefix, n_folds=N_FOLDS):
    n = len(full_dataset)
    indices = torch.randperm(n).tolist()
    fold_size = n // n_folds
    
    all_models = []
    
    for fold in range(n_folds):
        print(f"\n{'='*50}")
        print(f"  {model_prefix} - FOLD {fold+1}/{n_folds}")
        print(f"{'='*50}")
        
        # Split indices
        val_start = fold * fold_size
        val_end = val_start + fold_size if fold < n_folds - 1 else n
        val_idx = indices[val_start:val_end]
        train_idx = indices[:val_start] + indices[val_end:]
        
        # Create datasets with appropriate transforms
        train_ds = SingleMaskDataset(full_dataset.img_dir, full_dataset.mask_dir, transform=train_transform)
        train_ds.images = [full_dataset.images[i] for i in train_idx]
        train_ds._mask_lookup = full_dataset._mask_lookup
        
        val_ds = SingleMaskDataset(full_dataset.img_dir, full_dataset.mask_dir, transform=val_transform)
        val_ds.images = [full_dataset.images[i] for i in val_idx]
        val_ds._mask_lookup = full_dataset._mask_lookup
        
        train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
        val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
        
        print(f"  Train: {len(train_ds)}, Val: {len(val_ds)}")
        
        # Build model
        model = smp.Unet(
            encoder_name="efficientnet-b3",
            encoder_weights="imagenet",
            in_channels=1,
            classes=1,
        ).to(DEVICE)
        
        optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
        criterion = DiceBCELoss()
        scaler = torch.amp.GradScaler('cuda')
        
        best_val_loss = float('inf')
        save_path = OUT_DIR / f'{model_prefix}_fold{fold}.pth'
        
        for epoch in range(1, EPOCHS_PER_FOLD + 1):
            # Train
            model.train()
            train_loss = 0.0
            for imgs, masks in train_dl:
                imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
                optimizer.zero_grad()
                with torch.amp.autocast('cuda'):
                    preds = model(imgs)
                    loss = criterion(preds, masks)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                train_loss += loss.item() * imgs.size(0)
            train_loss /= len(train_ds)
            
            # Validate
            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for imgs, masks in val_dl:
                    imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
                    with torch.amp.autocast('cuda'):
                        preds = model(imgs)
                        loss = criterion(preds, masks)
                    val_loss += loss.item() * imgs.size(0)
            val_loss /= len(val_ds)
            
            scheduler.step(val_loss)
            current_lr = optimizer.param_groups[0]['lr']
            print(f"  Epoch {epoch:02d} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | LR: {current_lr:.6f}")
            
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                torch.save(model.state_dict(), save_path)
        
        # Load best and keep
        model.load_state_dict(torch.load(save_path, weights_only=True))
        model.eval()
        all_models.append(model)
        print(f"  Fold {fold+1} best val loss: {best_val_loss:.4f}")
    
    return all_models

# Train both model ensembles
print("\n" + "="*60)
print("  PHASE 1: Training Aponeurosis Ensemble")
print("="*60)
apo_models = train_kfold(apo_full, "apo_unet")

print("\n" + "="*60)
print("  PHASE 2: Training Fascicle Ensemble")
print("="*60)
fasc_models = train_kfold(fasc_full, "fasc_unet")

print(f"\nTraining complete! {len(apo_models)} Apo models + {len(fasc_models)} Fasc models ready.")


In [ ]:
def measure_geometry(apo_mask, fasc_mask, scale_mm_per_pixel=0.06979):
    ### 1. PENNATION ANGLE (PA) ###
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(fasc_mask, 8)
    angles, weights = [], []
    for i in range(1, n_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        if area < 50: continue
        pts = np.column_stack(np.where(labels == i)).astype(np.float64)
        if len(pts) < 5: continue
        cov = np.cov(pts.T)
        if cov.shape != (2, 2): continue
        evals, evecs = np.linalg.eigh(cov)
        principal = evecs[:, np.argmax(evals)]
        angle = np.degrees(np.arctan2(abs(principal[0]), abs(principal[1])))
        angles.append(angle)
        weights.append(float(area))
    pa_deg = np.average(angles, weights=weights) if angles else 15.0

    ### 2. MUSCLE THICKNESS (MT) ###
    n_apo, apo_labels, apo_stats, _ = cv2.connectedComponentsWithStats(apo_mask, 8)
    if n_apo >= 3:
        comps = sorted([(i, apo_stats[i, cv2.CC_STAT_AREA]) for i in range(1, n_apo)],
                       key=lambda x: x[1], reverse=True)
        c1, c2 = comps[0][0], comps[1][0]
        y1, y2 = np.mean(np.where(apo_labels == c1)[0]), np.mean(np.where(apo_labels == c2)[0])
        sup_c, deep_c = (c1, c2) if y1 < y2 else (c2, c1)

        sup_mask = (apo_labels == sup_c).astype(np.uint8)
        deep_mask = (apo_labels == deep_c).astype(np.uint8)

        sup_y = np.argmax(sup_mask[::-1], axis=0)
        sup_bottom = np.where(sup_mask.any(axis=0), sup_mask.shape[0] - 1 - sup_y, np.nan)
        deep_top = np.where(deep_mask.any(axis=0), np.argmax(deep_mask, axis=0), np.nan)

        thickness_profile = deep_top - sup_bottom
        valid = ~np.isnan(thickness_profile) & (thickness_profile > 0)
        mt_mm = np.nanmedian(thickness_profile[valid]) * scale_mm_per_pixel if np.sum(valid) > 50 else 20.0
    else:
        mt_mm = 20.0

    ### 3. FASCICLE LENGTH (FL) ###
    pa_rad = math.radians(max(pa_deg, 3.0))
    fl_mm = mt_mm / math.sin(pa_rad)

    pa_deg = float(np.clip(pa_deg, 3.0, 45.0))
    fl_mm = float(np.clip(fl_mm, 20.0, 200.0))
    mt_mm = float(np.clip(mt_mm, 5.0, 40.0))

    return pa_deg, fl_mm, mt_mm


In [ ]:
# --- Ensemble + 5x TTA Inference ---

def predict_ensemble_tta(models, img_tensor):
    """Average predictions across all fold models and 5 TTA variants."""
    all_preds = []
    
    tta_variants = [
        img_tensor,                                    # Original
        torch.flip(img_tensor, dims=[3]),               # Horizontal flip
        torch.flip(img_tensor, dims=[2]),               # Vertical flip
        img_tensor * 1.1,                               # Slightly brighter
        img_tensor * 0.9,                               # Slightly darker
    ]
    
    for model in models:
        for tta_input in tta_variants:
            with torch.amp.autocast('cuda'):
                pred = torch.sigmoid(model(tta_input))
            
            # Undo spatial flips
            if tta_input is tta_variants[1]:  # Was h-flipped
                pred = torch.flip(pred, dims=[3])
            elif tta_input is tta_variants[2]:  # Was v-flipped
                pred = torch.flip(pred, dims=[2])
            
            all_preds.append(pred)
    
    # Average all predictions (N_FOLDS * 5 TTA = 15 predictions)
    return torch.stack(all_preds).mean(dim=0)

# Collect test images
test_files = sorted([p for p in TEST_DIR.rglob('*') 
                     if p.is_file() and p.suffix.lower() in ['.tif', '.png', '.jpg']])
print(f"Running ensemble inference on {len(test_files)} test images...")
print(f"  Ensemble size: {len(apo_models)} Apo + {len(fasc_models)} Fasc models")
print(f"  TTA variants: 5 per model")
print(f"  Total predictions averaged per image: {len(apo_models) * 5} (Apo) + {len(fasc_models) * 5} (Fasc)")

results = []

with torch.no_grad():
    for test_path in tqdm(test_files, desc="Inference"):
        img_orig = cv2.imread(str(test_path), cv2.IMREAD_GRAYSCALE)
        if img_orig is None:
            continue
        orig_h, orig_w = img_orig.shape[:2]

        # Preprocess (same normalization as training)
        img_rs = cv2.resize(img_orig, (IMG_SIZE, IMG_SIZE))
        img_tensor = (img_rs.astype(np.float32) / 255.0 - 0.5) / 0.5
        img_tensor = torch.from_numpy(img_tensor).unsqueeze(0).unsqueeze(0).to(DEVICE)

        # Get ensemble+TTA predictions
        apo_pred = predict_ensemble_tta(apo_models, img_tensor)
        fasc_pred = predict_ensemble_tta(fasc_models, img_tensor)

        # Threshold and convert
        apo_np = (apo_pred.squeeze().cpu().numpy() > 0.5).astype(np.uint8) * 255
        fasc_np = (fasc_pred.squeeze().cpu().numpy() > 0.5).astype(np.uint8) * 255

        # Resize back to original dimensions
        apo_full = cv2.resize(apo_np, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)
        fasc_full = cv2.resize(fasc_np, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)

        # Extract geometry
        pa, fl, mt = measure_geometry(apo_full, fasc_full)
        image_id = test_path.name if test_path.suffix else f"{test_path.stem}.tif"

        results.append({
            'image_id': image_id,
            'pa_deg': round(pa, 3),
            'fl_mm': round(fl, 3),
            'mt_mm': round(mt, 3)
        })

# Save submission (COMMA separated!)
sub_df = pd.DataFrame(results)
sub_df.to_csv(OUT_DIR / 'submission.csv', sep=',', index=False)
print(f"\nSubmission saved! {len(sub_df)} rows")
print(f"  PA range: {sub_df['pa_deg'].min():.1f} - {sub_df['pa_deg'].max():.1f} deg")
print(f"  FL range: {sub_df['fl_mm'].min():.1f} - {sub_df['fl_mm'].max():.1f} mm")
print(f"  MT range: {sub_df['mt_mm'].min():.1f} - {sub_df['mt_mm'].max():.1f} mm")
print(sub_df.head())
